In [0]:
"""
id: source_0
template: source
name: source_0
position:
  x: 756
  y: 386
description:
  text: Load all data from a binary file at the specified path.
  hash: 38fccb8f
previewCodeHash: 6aeec2f4853607ec
previewMode: "1000"
config:
  file_source:
    path: /Volumes/personal_1/examples/files/rm_24_sparring_sam.pdf
    format: '"binaryFile"'
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/personal_1/examples/files/rm_24_sparring_sam.pdf",
        "format": "\"binaryFile\""
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]

In [0]:
"""
id: parse_document
template: sql
name: parse_document
parentId: group_4
position:
  x: 30
  y: 78
description:
  text: Extract content and parse it into a structured format.
  hash: 6322b644
previewCodeHash: c99b6d28a9a976f0
previewMode: "1000"
config:
  query: |-
    SELECT
      path,
      ai_parse_document(content) AS parsed
    FROM source_0
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT\n  path,\n  ai_parse_document(content) AS parsed\nFROM source_0"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "source_0",
            "df_name": "source_0"
        }
    ]
}
out = run(config, inputs, spark)
ctx["parse_document.result"] = out["result"]

In [0]:
"""
id: extract_text
template: sql
name: extract_text
parentId: group_4
position:
  x: 274.5725816694037
  y: 79.10195845218544
description:
  text: Extract and combine text content from parsed documents where there are no error statuses.
  hash: eaa42c9c
previewCodeHash: 401075613eafaa39
previewMode: "1000"
config:
  query: |-
    SELECT
      path,
      concat_ws('\n\n',
        transform(
          try_cast(parsed:document:elements AS ARRAY<VARIANT>),
          element -> try_cast(element:content AS STRING)
        )
      ) AS full_text
    FROM parse_document
    WHERE try_cast(parsed:error_status AS STRING) IS NULL
input:
  - node: parse_document
    input_port: data
    output_port: result
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT\n  path,\n  concat_ws('\\n\\n',\n    transform(\n      try_cast(parsed:document:elements AS ARRAY<VARIANT>),\n      element -> try_cast(element:content AS STRING)\n    )\n  ) AS full_text\nFROM parse_document\nWHERE try_cast(parsed:error_status AS STRING) IS NULL"
}
inputs = {
    "data": [
        ctx["parse_document.result"],
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "parse_document",
            "output_port": "result",
            "name": "parse_document",
            "df_name": "parse_document"
        },
        {
            "node": "source_0",
            "output_port": "data",
            "name": "source_0",
            "df_name": "source_0"
        }
    ]
}
out = run(config, inputs, spark)
ctx["extract_text.result"] = out["result"]

---
id: note_3
template: markdown
name: note_3
position:
  x: 1141.7230182676146
  y: 207.25701949557387
dimensions:
  width: 200
  height: 95
config:
  md: This is a test
---

---
id: group_4
template: group
name: group_4
position:
  x: 1007.8864771305109
  y: 343.1781968602073
dimensions:
  width: 504.5725816694037
  height: 204.10195845218544
---